In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [ ]:
class SimpleNN(nn.Module):
    """
    Трёхслойная полносвязная сеть с ReLU и Dropout для регуляризации.
    Вход: 16384 признака (пиксели 128х128).
    Выход: 5 классов (цифры от 0 до 4).
    """
    def __init__(self, input_size, num_classes):
        super().__init__()
        # Полносвязные слои
        self.fc1 = nn.Linear(input_size, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, num_classes)

        # Функция активации ReLU (нелинейность)
        self.relu = nn.ReLU()

        # Dropout — случайное отключение 20% нейронов во время обучения,
        # чтобы предотвратить переобучение.
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Прямой проход: последовательно применяем слои с активациями и dropout
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        # На выходе не применяем softmax, потому что CrossEntropyLoss
        # сама включает его в расчёт (логиты -> вероятности).
        x = self.fc3(x)
        return x

In [ ]:
model = SimpleNN(input_size=16384, num_classes=5)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Обучение
num_epochs = 15

for epoch in range(num_epochs):
    model.train()   # переключаем модель в train-режим (dropout активен)
    running_loss = 0.0

    for inputs, labels in train_loader:
        # Обнуляем градиенты, накопленные за предыдущий батч
        optimizer.zero_grad()

        # Прямой проход: получаем логиты
        outputs = model(inputs)

        # Вычисляем значение функции потерь между предсказаниями и истинными метками
        loss = criterion(outputs, labels)

        # Обратное распространение (вычисляем градиенты)
        loss.backward()

        # Обновляем веса сети (один шаг оптимизатора)
        optimizer.step()

        # Суммируем потери за батч (для вывода среднего по эпохе)
        running_loss += loss.item()

    # ---------- РЕЖИМ ОЦЕНКИ (на тестовой выборке) ----------
    model.eval()    # переключаем в eval-режим (dropout выключается)
    correct = 0
    total = 0

    # Отключаем вычисление градиентов — это экономит память и ускоряет расчёты
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)                # получаем логиты
            _, pred = torch.max(outputs, 1)        # выбираем класс с максимальной вероятностью
            total += labels.size(0)                # общее количество примеров в батче
            correct += (pred == labels).sum().item()  # число правильных ответов

    # Считаем точность (accuracy) в процентах
    acc = 100 * correct / total

    # Печатаем статистику по эпохе
    print(f'Epoch {epoch+1:2d} | Loss: {running_loss/len(train_loader):.4f} | Test Acc: {acc:.2f}%')

# ----------------------------------------------------------------------
# После завершения обучения модель готова к использованию.
# Можно сохранить веса через torch.save(model.state_dict(), 'model.pth')
# и загружать позже.
# ----------------------------------------------------------------------

In [ ]:
#Сохранение модели
import joblib

torch.save(model.state_dict(), 'model_weights.pth')
joblib.dump(scaler, 'scaler.save')

print("Модель и scaler сохранены.")

In [ ]:
loaded_model = SimpleNN(input_size=16384, num_classes=5)
loaded_model.load_state_dict(torch.load('model_weights.pth'))

loaded_model.eval()

loaded_scaler = joblib.load('scaler.save')

print("Модель и scaler загружены.")